<a href="https://colab.research.google.com/github/kimhozzi/Kleague_Crawling/blob/main/crawl_team.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install beautifulsoup4 requests selenium lxml openpyxl

In [ ]:
#K리그 데이터 포털은 옛날 방식인 <frameset> 구조로 되어 있습니다. 껍데기(URL 불변) 안에 실제 내용이 담긴 페이지가 따로 있는 구조입니다.

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import pandas as pd

# 웹드라이버를 설정합니다. (여기서는 Chrome을 사용합니다.)
# 웹드라이버의 경로를 자신의 시스템에 맞게 설정해주세요.
options = Options()
options.page_load_strategy = 'eager'
driver = webdriver.Chrome(options=options)

# 웹사이트에 접속합니다.
driver.get('https://data.kleague.com')

# Find frameset element and get its children
frames = driver.find_elements("tag name", 'frame')

for frame in frames:
    if 'https://portal.kleague.com' in frame.get_attribute('src'):
        # redirect the browser to the frame url
        driver.get(frame.get_attribute('src'))
    else:
        # do nothing
        pass

In [ ]:
import re

def extract_onclick_data(onclick):
    if "moveMainFrameMcPlayer" in onclick:
        match = re.search(r"moveMainFrameMcPlayer\('(.+?)','(.+?)','(.+?)'\)", onclick)
        if match:
            return dict(zip(['menuCd', 'playerId', 'teamId'], match.groups()))
    elif "moveMainFrame" in onclick:
        match = re.search(r"moveMainFrame\('(.+?)'\)", onclick)
        if match:
            return {'menuCd': match.group(1)}
    return {}

In [ ]:
# JavaScript를 실행하여 페이지를 이동합니다.
# driver.execute_script("moveMainFrame('0011')")
# driver.execute_script("moveMainFrame('0194')")  -> team
### !! click 아닌 excute_script 사용 가능성 물어보기
driver.execute_script("moveMainFrame('0194')")

In [ ]:
## TEAM , YEARS -> 이거 상수라서 대문자화 해도 될듯 ㅇㅇ 2025 결과 생성됨 반영. dict로 js변수, 한글로 만듦

team_dict = {
    'K01': '울산', 'K02': '수원삼성', 'K03': '포항', 'K04': '제주',
    'K05': '전북', 'K07': '부산', 'K08': '성남', 'K09': '서울',
    'K10': '대전', 'K17': '대구', 'K18': '인천', 'K21': '강원',
    'K22': '광주', 'K27': '안양', 'K29': '수원FC', 'K35': '김천'
}

years = ['2021', '2022', '2023', '2024','2025'] # 2025는 아직 데이터가 없으므로 제외


In [ ]:
### 여기까지 완료  ....

### 과제   --->  이거 click 같이 쓸 수 있나 driver.execute_script랑 onClick()이랑 같이 쓸 수 있는지???

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import Select  # Select 모듈 추가
from bs4 import BeautifulSoup
import pandas as pd
import time

# ... (웹드라이버 설정 및 프레임 이동 코드는 위와 동일) ...

# 1. 초기 페이지 이동 (기록실 등)
driver.execute_script("moveMainFrame('0194')")

# 2. 데이터 수집 준비
team_dict = {
    'K01': '울산', 'K02': '수원삼성', 'K03': '포항', 'K04': '제주',
    'K05': '전북', 'K07': '부산', 'K08': '성남', 'K09': '서울',
    'K10': '대전', 'K17': '대구', 'K18': '인천', 'K21': '강원',
    'K22': '광주', 'K27': '안양', 'K29': '수원FC', 'K35': '김천'
}
years = ['2021', '2022', '2023', '2024']
all_teams_data = []

# [핵심 수정] 함수화: 안전하게 값 선택하기
def safe_select_option(driver, element_id, value):
    try:
        # 해당 ID가 화면에 뜰 때까지 최대 10초 대기
        element = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.ID, element_id))
        )
        # JS 대신 Selenium 내장 Select 기능 사용 (더 안정적)
        select = Select(element)
        select.select_by_value(value)
        return True
    except Exception as e:
        print(f"  ERROR: '{element_id}' 요소를 찾을 수 없거나 값을 설정할 수 없습니다.")
        return False

# 3. 루프 시작
for year in years:
    for team_id, team_name in team_dict.items():
        print(f"[{year}] {team_name} 데이터 조회 시도...")

        try:
            # (1) 연도 선택 (JS 강제 실행 대신 대기 후 선택)
            if not safe_select_option(driver, 'selectYear', year):
                continue # 실패시 다음 팀으로

            # (2) 팀 선택
            if not safe_select_option(driver, 'selectTeamId', team_id):
                continue

            # (3) 조회 버튼 클릭
            # 버튼도 로딩될 때까지 기다림
            search_btn = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable((By.ID, 'btnSearch'))
            )
            search_btn.click() # JS 대신 .click() 사용

            # (4) 데이터 로딩 대기
            # 테이블이 바뀌는 것을 기다려야 함.
            time.sleep(1.5) # 간단하게 sleep 유지 (네트워크 느리면 늘리세요)

            # (5) 데이터 파싱
            html = driver.page_source
            if "데이터가 존재하지 않습니다" in html:
                print(f"   -> 데이터 없음 (Pass)")
                continue

            # 테이블 찾기 (ID가 없을 수도 있으니 예외처리 강화)
            try:
                # pandas가 테이블을 찾도록 시도
                dfs = pd.read_html(html)

                # 보통 우리가 원하는 큰 데이터는 행(row)이 가장 많은 테이블일 확률이 높음
                target_df = None
                max_rows = 0

                for df in dfs:
                    if len(df) > max_rows:
                        max_rows = len(df)
                        target_df = df

                if target_df is not None and not target_df.empty:
                    target_df['시즌'] = year
                    target_df['팀명'] = team_name
                    all_teams_data.append(target_df)
                    print(f"   -> 수집 성공: {len(target_df)}행")
                else:
                    print("   -> 유효한 테이블을 찾지 못함")

            except ValueError:
                print("   -> HTML 내에 테이블 태그가 없음")

        except Exception as e:
            print(f"   -> 알 수 없는 에러 발생: {e}")
            # 에러 발생 시 페이지 새로고침 등으로 복구 시도 가능
            driver.refresh()
            time.sleep(2)
            driver.execute_script("moveMainFrame('0194')") # 페이지 원상복구
            continue

# 4. 저장
if all_teams_data:
    final_result = pd.concat(all_teams_data, ignore_index=True)
    file_name = "K리그_통합데이터_result.csv"
    final_result.to_csv(file_name, index=False, encoding='utf-8-sig')
    print(f"\n저장 완료: {file_name}")
else:
    print("\n수집된 데이터가 없습니다.")

driver.quit()

[2021] 울산 데이터 조회 시도...


C:\Users\oi\AppData\Local\Temp\ipykernel_16920\2412368256.py:75: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(html)


   -> 수집 성공: 38행
[2021] 수원삼성 데이터 조회 시도...


C:\Users\oi\AppData\Local\Temp\ipykernel_16920\2412368256.py:75: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(html)


   -> 수집 성공: 38행
[2021] 포항 데이터 조회 시도...


...

   -> 수집 성공: 40행
[2024] 안양 데이터 조회 시도...
  ERROR: 'selectTeamId' 요소를 찾을 수 없거나 값을 설정할 수 없습니다.
[2024] 수원FC 데이터 조회 시도...


C:\Users\oi\AppData\Local\Temp\ipykernel_16920\2412368256.py:75: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(html)


   -> 수집 성공: 40행
[2024] 김천 데이터 조회 시도...


C:\Users\oi\AppData\Local\Temp\ipykernel_16920\2412368256.py:75: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(html)


   -> 수집 성공: 40행

저장 완료: K리그_통합데이터_result.csv
